In [1]:
%%html
<script>//Needs to be "nbsphinx" : "hidden"
MathJax.Hub.Config({
    TeX: { equationNumbers: {autoNumber: 'AMS'} }});
MathJax.Hub.Queue(
  ["resetEquationNumbers", MathJax.InputJax.TeX],
  ["PreProcess", MathJax.Hub],
  ["Reprocess", MathJax.Hub]
);
</script>

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Due to a Jupyter bug, the %matplotlib inline "magic" command needs to be in 
# its own cell. See:
# https://github.com/jupyter/notebook/issues/3385

%matplotlib inline

In [4]:
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

# Adjust default font size so that labels are readable in plots
font = {'size': 16}
matplotlib.rc('font', **font)
matplotlib.rcParams['savefig.dpi'] = 200
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['figure.figsize'] = [7, 3.5]

# Model Fitting Part III - Optimization Techniques


We have seen that in general we will need to maximize a likelihood or a Bayesian posterior.

The likelihood may have a least squares ($\chi^2$) form, or it may be more complicated.

There are two classes of algorithms: those that find a local minimum, and those that attempt to find a global one.

## Finding the _Local_ Minimum

There are several methods for finding the local minimum

### [Nelder-Mead](https://en.wikipedia.org/wiki/Nelder%E2%80%93Mead_method) or Simplex

Reliable but slow, does not need derivatives. A simplex is a an M+1 vertex object in an M dimensional space, e.g. a triangle in 2D, tetrahedron in 3D.

[Simplex](https://upload.wikimedia.org/wikipedia/commons/d/de/Nelder-Mead_Himmelblau.gif)


### Conjugate gradient or Powell's method

Faster for well behaved functions. Does not need derivatives.



A comparison of the convergence of gradient desc|ent with optimal step size (in green) and conjugate vector (in red) for minimizing a quadratic function associated with a given linear system. Conjugate gradient, assuming exact arithmetic, converges in at most $n$ steps, where $n$ is the size of the matrix of the system (here $n = 2$).

### Levenberg-Marquardt

A combination of Gauss-Newton and gradient descent. 

This is the method used by ``scipy.optimize.leastsq`` and ``scipy.optimize.curve_fit``



Many of these algorithms are built into the [lmfit](https://lmfit.github.io/lmfit-py/index.html)
package which is a wrapper around [scipy.optimize.minimize](https://docs.scipy.org/doc/scipy/reference/optimize.html) and [scipy.optimize.leastsq](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.leastsq.html), which itself is wrapped around the FORTRAN minpack routines.

``lmfit`` has two major ease-of-use advantages:

1. easily set allowed ranges (minima and maxima) for individual parameters
2. easily freeze a parameter or allow it to vary


Let's simulate the detection of a line above a constant background (could be an emission line in a galaxy spectrum, or the Higgs boson)

The line will have a number of counts, a position and a width but its shape will be assumed to be Gaussian.

Make some fake data

In [5]:
import numpy.random
import scipy.stats

def gendata(n,frac,pos,width):

    # make some fake uniform data
    duniform = numpy.random.rand(int((1.-frac)*n))
    
    # Add a Gaussian peak
    rv = scipy.stats.norm(loc=pos, scale=width)
    dline = rv.rvs(int(frac*n))
               
    data = np.hstack((duniform,dline))
    return data

In [ ]:
def plot_data_model(data, nbins=1000, frac=None, pos=None, width=None):
    
    ntot = len(data) 
    counts, edges, patches = plt.hist(data, bins=nbins)
    # make a smooth model curve, coitrrectly normalized
    xhi = edges[-1]
    xlo = edges[0]
    binwidth = (xhi-xlo)/nbins

    
    normal = scipy.stats.norm(loc=pos, scale=width)

    x = np.linspace(xlo, xhi, 1000)
    pdf  = (1.-frac)/(xhi-xlo) + frac*(normal.pdf(x))
    
    plt.plot(x, pdf*ntot*binwidth)
    return (counts, edges)

If we had amazing data the results might look like this:

In [ ]:
n = 500_000
frac = 0.2
pos = 0.5
width= 0.02
data= gendata(n,frac,pos,width)

#counts, edges, patches = plt.hist(data, bins=1000)
(counts, edges) = plot_data_model(data, frac=frac, pos=pos, width=width)

More often we have poorer data and have to deal with something like this:

In [ ]:
n = 500
frac = 0.2
pos = 0.5
width= 0.02
data= gendata(n,frac,pos,width)

(counts, edges) = plot_data_model(data, nbins=10, frac=frac, pos=pos, width=width)

Note: for the rest of this notebook and the next you may need to install some non-standard packages:

```
conda install -c conda-forge lmfit
conda install -c conda-forge emcee
conda install -c conda-forge corner
```


In [ ]:
import lmfit

params = lmfit.Parameters()
params.add('frac', value=0.3, min=0., max=1.0)
params.add('pos', value=0.6, min=0.2, max=0.8)
params.add('width', value=0.2, min=1.e-6, max=0.3)
params

In [ ]:
params.pretty_print()

In [ ]:
import scipy.stats
# A fitting function for the histogram

def residual(params, counts, edges):

    ntot = np.sum(counts)

    # unpack parameters: extract .value attribute for each parameter
    val = params.valuesdict()
    mean = val['pos']
    std = val['width']
    rv = scipy.stats.norm(mean, std)
    
    linecounts = (rv.cdf(edges[1:]) - rv.cdf(edges[:-1])) * val['frac'] * ntot

    fracbinwidths = (edges[1:] - edges[:-1]) / (edges[-1] - edges[0])
    bkgdcounts = (1. - val['frac']) * ntot * fracbinwidths

    model = bkgdcounts + linecounts

    # Return the residual, normalized by the uncertainties
    # For a Poisson distribution, the mean and variance are the same
    return (counts - model) / np.sqrt(model)

In [ ]:
fitter = lmfit.Minimizer(residual, params, fcn_args=(counts, edges))
result = fitter.minimize(method='nelder-mead')

# result is a MinimizerResult object. It has several useful data/methods
result.params.pretty_print()


In [ ]:
val = result.params.valuesdict()
plot_data_model(data, nbins=10, frac=val['frac'], pos=val['pos'], width=val['width'])

Notice that there are no uncertainties.

In ``lmfit`` only the ``leastsq`` method yields uncertainties. 

So once you have found the minimum by any method, rerun starting from there but using ``leastsq`` 

In [ ]:
result = lmfit.minimize(residual, params, args=((counts, edges)), method='leastsq') 
# leastsq is Levenberg-Marquardt

print(result.nfev, "function evaluations")
result.params.pretty_print()
print("Chi^2:", result.chisqr, "AIC:", result.aic, "BIC:", result.bic)

In [ ]:
print(result.covar)

Another way to print results and correlation matrix

In [ ]:
print(lmfit.fit_report(result.params))

Now suppose we wanted to fix a parameter to some value. With ``scipy.optimize.curve_fit``, you would have to rewrite a whole lot of code. 

It's a lot easier with `lmfit`

In [ ]:
# Fix the parameter pos=0.5
params['pos'].value = 0.5
params['pos'].vary = False

fitter = lmfit.Minimizer(residual, params, fcn_args=(counts, edges))
result = fitter.minimize(method='leastsq')
print(result.nfev)
result.params.pretty_print()
print("Chi^2:", result.chisqr, "AIC:", result.aic, "BIC:", result.bic)

The fit is working, but is not finding the correct result for the width due to the binning


In [ ]:
# This is the log-likelihood probability for the sampling.

def prob(data, frac, pos, width):

    froznorm = scipy.stats.norm(loc=pos, scale=width)
    pgauss = frac*froznorm.pdf(data)

    xrange = 1.
    puniform = (1.-frac)/xrange
    prob = puniform + pgauss 
    # note model is automatically normalized
    return prob

def lnprob(param, data):
    vals = param.valuesdict()
    probs = prob(data, vals['frac'], vals['pos'], vals['width'])

    return np.sum(np.log(probs))

def lnprob_arr(arr, data):
    probs = prob(data, arr[0], arr[1], arr[2])
    
    return np.sum(np.log(probs))

Double check that our probability integrates to 1.

In [ ]:
import scipy.integrate

print(frac,pos,width)
integ, err = scipy.integrate.quad(prob, 0., 1., args=(frac,pos,width))
print(integ)

In [ ]:
# Note the keyword argument fcn_args
params['frac'].value=0.5
params['pos'].value=0.6
params['pos'].vary = True
params['width'].value=0.2

In [ ]:
# for a scalar cost function we will minimize
# so we need the function to be negative
nll = lambda *args: -lnprob(*args)
minim = lmfit.Minimizer(nll, params, fcn_args=((data,)))

result = minim.scalar_minimize(method='Nelder-Mead')
lmfit.printfuncs.report_fit(result.params, min_correl=0.01)


The problem with this method is that it doesn't measure uncertainties (at least not yet in ``lmfit``, but should work with [``scipy.optimize.curve_fit``](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html))

An alternative is to use Monte-Carlo Markov Chains, or MCMC. We will describe how these in more detail work later. 

In [ ]:
def plotchains(chain, labels=None):
    shape = chain.shape
    (nsteps, nwalkers, npars) = list(shape)
    print("Chain shape: ", nsteps, "steps,", nwalkers, "walkers,", npars, "params." )
    for npar in range(npars):
        # start a new plot
        fig, ax = plt.subplots()
        for nwalk in range(nwalkers):
            plt.plot(range(nsteps), chain[:,nwalk,npar], lw=0.7)
        if labels:
            plt.xlabel('Steps')
            plt.ylabel(labels[npar])
                

In [ ]:
# for emcee, we *maximise*, so the function is just ln(posterior)
minim = lmfit.Minimizer(lnprob, params, fcn_args=((data,)),float_behavior='posterior')
emcee_result = minim.emcee(burn=0, steps=500, thin=1, params=minim.params)

Each line in the following plot represents a chain of parameters

In [ ]:
plotchains(emcee_result.chain, labels=['frac','pos','width'])

We see that it takes some number of steps for the parameters to settle down. This is called the "burn-in" phase. 

In this example, it looks like takes about 150 steps to burn in.

``emcee`` allows you to discard a burn-in phase.

In [ ]:
# for emcee where we will *maximise*, so the function is just ln(post)
minim = lmfit.Minimizer(lnprob, params, fcn_args=((data,)),float_behavior='posterior')
emcee_result = minim.emcee(burn=250, steps=5000, thin=1, params=minim.params)

In [ ]:
plotchains(emcee_result.chain)

Now the amazing thing is that the MCMC chains sample the probability distribution of the parameters!

In [ ]:
import corner
fig = corner.corner(emcee_result.flatchain, labels=emcee_result.var_names, 
                    quantiles=[0.16, 0.5, 0.84],show_titles=True)
                    #truths=list(emcee_result.params.valuesdict().values()))

In [ ]:
quant = [0.16, 0.5, 0.84]
for param in ['frac', 'pos', 'width']:
    low, median, high = np.quantile(emcee_result.flatchain[param], quant)
    # mean = np.mean(emcee_result.flatchain[param])
    print(f'{param:10s} {median:10.4g} +{high-median:10.4g} -{median-low:1.4g}')
